# Experiment Runner Notebook

This notebook automates the experiment lineup for Google Colab. It:
- Installs dependencies and prepares the repo
- Provides a runner to launch training runs (uses the repo's `core_ml/train/train.py`)
- Runs the experiment grid you requested (attention variants, windows, positional tests)
- After each run it performs an evaluation pass and appends results to a JSON file so progress is incremental.

Notes:
- The notebook expects the repository to be available in the Colab VM (e.g., mounted from Drive). Update `REPO_DIR` if needed.
- Heavy training should be run on an appropriate GPU runtime (Colab Pro recommended).

In [ ]:
# Colab setup: change REPO_DIR to where you mount the repository (Drive or upload)
import os, sys
import subprocess
REPO_DIR = '/content/SAiDL-Summer-Assignment-2026'  # update if different

if not os.path.exists(REPO_DIR):
    print('Repo not found at', REPO_DIR)
    print('Please mount your Drive or upload the repo to Colab and set REPO_DIR accordingly.')
else:
    os.chdir(REPO_DIR)
    if os.path.exists('requirements.txt'):
        print('Installing requirements...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
    else:
        print('No requirements.txt found; install dependencies manually if needed.')
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print('Working directory:', os.getcwd())

## Runner utilities
The next cell defines helper functions to run training, evaluate a saved checkpoint, and record results incrementally to `experiments_results.json`.

In [ ]:
def evaluate_checkpoint(overrides: Dict[str, Any], output_dir: str, device: str = 'cuda') -> Dict[str, Any]:
    overrides_json = json.dumps(overrides).replace("'", "\\'")
    runner = textwrap.dedent("""
    import os, sys, json, torch, time
    from omegaconf import OmegaConf
    repo_root = r'{repo_root}'
    sys.path.insert(0, repo_root)
    from core_ml.train.dataset import prepare_dataloaders
    from core_ml.train.trainer import Trainer
    from core_ml.train.train import build_model
    overrides = json.loads(r'''{overrides_json}''')
    cfg_path = os.path.join(repo_root, 'core_ml', 'configs', 'config.yaml')
    cfg = OmegaConf.load(cfg_path)
    for k, v in overrides.items():
        parts = k.split('.')
        d = cfg
        for p in parts[:-1]:
            if p not in d:
                d[p] = {}
            d = d[p]
        d[parts[-1]] = v
    cfg.output_dir = r'{output_dir}'
    train_loader, val_loader = prepare_dataloaders(cfg)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(cfg).to(device)
    ckpt = os.path.join(cfg.output_dir, 'best_model.pt')
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state)
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()
    total_loss = 0.0
    total_tokens = 0
    t_start = time.perf_counter()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            n_toks = x.numel()
            total_loss += loss.item() * n_toks
            total_tokens += n_toks
    elapsed = time.perf_counter() - t_start
    avg_loss = total_loss / max(total_tokens, 1)
    import math
    val_ppl = float(math.exp(min(avg_loss, 20.0)))
    inf_tok_per_sec = total_tokens / max(elapsed, 1e-9)
    peak_mb = 0.0
    if torch.cuda.is_available():
        peak_mb = torch.cuda.max_memory_allocated(device) / 1e6
    out = {'val_loss': float(avg_loss), 'val_ppl': val_ppl, 'inference_tok_per_sec': float(inf_tok_per_sec), 'peak_gpu_mb': float(peak_mb)}
    print(json.dumps(out))
    """).format(
        repo_root=REPO_ROOT.replace('\\', '\\\\'),
        overrides_json=overrides_json,
        output_dir=output_dir.replace('\\', '\\\\'),
    )
    proc = subprocess.Popen([PYTHON_BIN, '-c', runner], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = proc.communicate()
    if proc.returncode != 0:
        raise RuntimeError(f"Eval script failed:\n{stderr}\n{stdout}")
    return json.loads(stdout.strip())


def record_result(params: Dict[str, Any], train_info: Dict[str, Any], eval_info: Dict[str, Any], results_path: str = RESULTS_PATH):
    rec = {'timestamp': time.time(), 'params': params, 'train_info': train_info, 'eval_info': eval_info}
    append_json(results_path, rec)
    print('Recorded result to', results_path)
    git_commit_results(results_path)


## Experiment grids and helper cells
The following cells contain the experiment grids; run them one-by-one.
Recommended batch-size mapping (tune to your GPU): {512:8, 1024:4, 2048:2, 4096:1}.
Each run writes its results to `experiments_results.json` so progress is incremental.

In [ ]:
# --- Baseline training (Colab) ---
# This cell runs the baseline `vanilla_mha` model with sequence length 1024.
# It is safe to run on Colab (will use the mounted repo) and will record results
# to `experiments_results.json` in the repo root.
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}

params = {
    'seq_len': 1024,
    'batch_size': bs_map[1024],
    'attention': 'vanilla_mha',
    'num_epochs': 1,          # set higher for full training
    'early_stop_steps': 2000, # optional short-circuit for faster runs
}

run_name = 'baseline_seq1024'
out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
os.makedirs(out_dir, exist_ok=True)
ov = {
    'dataset.seq_len': params['seq_len'],
    'dataset.batch_size': params['batch_size'],
    'attention.name': params['attention'],
    'training.num_epochs': params['num_epochs'],
    'training.early_stop_steps': params['early_stop_steps'],
    'output_dir': out_dir,
}

print('Starting baseline run:', run_name)
train_res = run_training(ov, run_name, timeout=60*60*6)
print('Train finished. Success:', train_res['ok'])

try:
    eval_info = evaluate_checkpoint(ov, out_dir)
except Exception as e:
    eval_info = {'error': str(e)}

record_result(params, train_res, eval_info)
print('Evaluation:', eval_info)

In [ ]:
# --- GQA grid: num_kv_heads in [2,1] across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
gqa_heads = [2, 1]
seq_grid = [512, 1024, 2048, 4096]

for heads in gqa_heads:
    for seq in seq_grid:
        params = {
            'seq_len': seq,
            'batch_size': bs_map[seq],
            'attention': 'gqa',
            'num_kv_heads': heads,
            'num_epochs': 1,
            'early_stop_steps': 2000,
        }
        run_name = f'gqa_heads{heads}_seq{seq}'
        out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': params['seq_len'],
            'dataset.batch_size': params['batch_size'],
            'attention.name': params['attention'],
            'attention.num_kv_heads': params['num_kv_heads'],
            'training.num_epochs': params['num_epochs'],
            'training.early_stop_steps': params['early_stop_steps'],
            'output_dir': out_dir,
        }

        print('\n=== RUN:', run_name)
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
        try:
            eval_info = evaluate_checkpoint(ov, out_dir)
        except Exception as e:
            eval_info = {'error': str(e)}
        record_result(params, train_res, eval_info)
        print('Done', run_name, '->', eval_info)

In [ ]:
# --- Sparse attention experiments across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'sparse_attention',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'sparse_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = evaluate_checkpoint(ov, out_dir)
    except Exception as e:
        eval_info = {'error': str(e)}
    record_result(params, train_res, eval_info)
    print('Done', run_name, '->', eval_info)

In [ ]:
# --- Sparse attention experiments across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'sparse_attention',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'sparse_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = evaluate_checkpoint(ov, out_dir)
    except Exception as e:
        eval_info = {'error': str(e)}
    record_result(params, train_res, eval_info)
    print('Done', run_name, '->', eval_info)

In [ ]:
# --- Positional encoding experiments (RoPE, ALiBi, Relative) with Ltrain=512 and extrapolation evals ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
pos_variants = ['rope', 'alibi', 'relative']
Ltrain = 512
Ltests = [512, 1024, 2048]

for pos in pos_variants:
    params = {
        'seq_len': Ltrain,
        'batch_size': bs_map[Ltrain],
        'attention': 'vanilla_mha',
        'positional': pos,
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'pos_{pos}_train{Ltrain}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'positional.name': params['positional'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'output_dir': out_dir,
    }

    print('\n=== TRAIN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train finished. Success:', train_res['ok'])

    for Ltest in Ltests:
        eval_params = params.copy()
        eval_params['seq_len'] = Ltest
        eval_ov = {
            'dataset.seq_len': eval_params['seq_len'],
            'dataset.batch_size': bs_map[Ltrain],
            'attention.name': eval_params['attention'],
            'positional.name': eval_params['positional'],
            'output_dir': out_dir,
        }
        print(f'\n=== EVAL {pos} trained on {Ltrain} | test_seq={Ltest}')
        try:
            eval_info = evaluate_checkpoint(eval_ov, out_dir)
        except Exception as e:
            eval_info = {'error': str(e)}
        record_result({'train_run': run_name, 'eval_seq_len': Ltest, 'positional': pos}, train_res, eval_info)
        print('Eval result:', eval_info)

In [ ]:
# --- Softmax (vanilla multi-head) across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'vanilla_mha',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'softmax_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = evaluate_checkpoint(ov, out_dir)
    except Exception as e:
        eval_info = {'error': str(e)}
    record_result(params, train_res, eval_info)
    print('Done', run_name, '->', eval_info)

In [ ]:
# --- Sliding window experiments (window sizes seq/2, seq/4, seq/8) across seq grid ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    for div in [2, 4, 8]:
        window = max(1, seq // div)
        params = {
            'seq_len': seq,
            'batch_size': bs_map[seq],
            'attention': 'sliding_window',
            'window_size': window,
            'num_epochs': 1,
            'early_stop_steps': 2000,
        }
        run_name = f'sliding_seq{seq}_w{window}'
        out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': params['seq_len'],
            'dataset.batch_size': params['batch_size'],
            'attention.name': params['attention'],
            'attention.window_size': params['window_size'],
            'training.num_epochs': params['num_epochs'],
            'training.early_stop_steps': params['early_stop_steps'],
            'output_dir': out_dir,
        }

        print('\n=== RUN:', run_name)
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
        try:
            eval_info = evaluate_checkpoint(ov, out_dir)
        except Exception as e:
            eval_info = {'error': str(e)}
        record_result(params, train_res, eval_info)
        print('Done', run_name, '->', eval_info)

## Complete experiment sequence
This notebook now contains the full requested lineup, one cell at a time:
1. Baseline with `seq_len=1024`
2. GQA grid with `num_kv_heads=2` and `1` across `[512, 1024, 2048, 4096]`
3. Sliding window experiments with window sizes `seq/2`, `seq/4`, `seq/8` across `[512, 1024, 2048, 4096]`
4. Softmax (vanilla multi-head) across `[512, 1024, 2048, 4096]`
5. Sparse attention across `[512, 1024, 2048, 4096]`
6. Positional encoding tests using RoPE, ALiBi, and Relative, plus extrapolation evaluation on `512`, `1024`, and `2048`.

Each run writes its results to `experiments_results.json`, so if the Colab session resets you can re-run the next cells without losing previous data.